In [15]:
import pandas as pd
import numpy as np
import string
import re
import wandb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [17]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [18]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [19]:
cols = ["prompt","A","B","C","D","E"]

for col in cols:
    train[col] = train[col].apply(clean)
    test[col] = test[col].apply(clean)

In [20]:
labels = ["A","B","C","D","E"]

texts = []
targets = []
question_ids = []

In [21]:
for i in range(len(train)):
    
    prompt = train.loc[i, "prompt"]
    answer = train.loc[i, "answer"]
    
    for label in labels:
        option = train.loc[i, label]
        
        text = prompt + " [SEP] " + option
        texts.append(text)
        
        if label == answer:
            targets.append(1)
        else:
            targets.append(0)
            
        question_ids.append(i)

In [22]:
print(len(texts))

10000


In [23]:
question_indices = list(range(len(train)))

train_q, val_q = train_test_split(
    question_indices,
    test_size=0.2,
    random_state=42
)

In [24]:
train_texts = []
train_targets = []

val_texts = []
val_targets = []

In [25]:
for idx, qid in enumerate(question_ids):
    
    if qid in train_q:
        train_texts.append(texts[idx])
        train_targets.append(targets[idx])
    else:
        val_texts.append(texts[idx])
        val_targets.append(targets[idx])

In [26]:
print(len(train_texts))
print(len(train_targets))

print(len(val_texts))
print(len(val_targets))

8000
8000
2000
2000


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

In [ ]:
MAX_LEN = 128

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

In [ ]:
import torch
from torch.utils.data import Dataset

class MCQDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.float
        )

        return item

In [ ]:
train_dataset = MCQDataset(
    train_encodings,
    train_targets
)

val_dataset = MCQDataset(
    val_encodings,
    val_targets
)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)

In [ ]:
import torch.nn as nn

In [ ]:
class BiLSTMAttention(nn.Module):

    def __init__(self,
                 vocab_size,
                 embed_dim=100,
                 hidden_dim=128):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(
            hidden_dim * 2,
            1
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            hidden_dim * 2,
            1
        )

    def forward(self, input_ids):

        embedded = self.embedding(input_ids)

        lstm_output, _ = self.lstm(embedded)

        attention_scores = self.attention(
            lstm_output
        ).squeeze(-1)

        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        context = torch.sum(
            lstm_output *
            attention_weights.unsqueeze(-1),
            dim=1
        )

        context = self.dropout(context)

        output = self.fc(context)

        return output.squeeze()

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = BiLSTMAttention(
    vocab_size=tokenizer.vocab_size
).to(device)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)